# Binned QML Power Spectrum Estimation

This notebook demonstrates the binning capability in QUBE. Binning sums
derivative matrices within multipole bins before computing the Fisher matrix
and QML estimates, reducing the parameter space from `n_ell` to `n_bins`.

Key property: by linearity of the trace, native binned Fisher equals
post-hoc binned Fisher: `F_binned == P @ F_unbinned @ P^T`.

In [1]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import yaml

from cosmocore import Bins
from qube import Fisher, Spectra

## 1. Setup

Resolve test data config paths.

In [2]:
QUBE_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))


def resolve_config(config_rel, overrides=None):
    full_path = os.path.join(QUBE_ROOT, config_rel)
    with open(full_path) as f:
        config = yaml.safe_load(f)
    for key, value in config.items():
        if isinstance(value, str):
            clean = value.lstrip("../")
            if clean.startswith("tests/") or clean.startswith("scripts/"):
                config[key] = os.path.join(QUBE_ROOT, clean)
    if overrides:
        config.update(overrides)
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False)
    yaml.dump(config, tmp, default_flow_style=False)
    tmp.close()
    return tmp.name


config_file = resolve_config("tests/data/nside4/T/config.yaml")
print(f"Config: {config_file}")

Config: /tmp/tmp5gw50ou9.yaml


## 2. Unbinned Analysis (delta_ell = 1)

Standard per-multipole QML estimation.

In [3]:
fisher_unb = Fisher(config_file)
fisher_unb.run()

spectra_unb = Spectra(config_file, fisher=fisher_unb)
spectra_unb.run()

F_unb = fisher_unb.get_fisher_matrix()
cl_unb = spectra_unb.get_power_spectra()
err_unb = spectra_unb.get_error_bars()
ell_unb = np.arange(2, spectra_unb.params.lmax + 1)

print(f"Unbinned: {F_unb.shape[0]} multipoles (ell = {ell_unb[0]}..{ell_unb[-1]})")
print(f"Power spectra shape: {cl_unb.shape}")
print(f"Fisher matrix shape: {F_unb.shape}")

2026-04-14 23:00:31 | fisher       | INFO     | Starting Fisher matrix analysis pipeline


2026-04-14 23:00:32 | fisher       | INFO     | --------------------------------------------------------------------------------
2026-04-14 23:00:32 | fisher       | INFO     | Fisher matrix computation completed
Unbinned: 7 multipoles (ell = 2..8)
Power spectra shape: (50, 7)
Fisher matrix shape: (7, 7)


## 3. Binned Analysis (delta_ell = 2)

Two ways to enable binning:

**Option A** — Python API:
```python
bins = Bins.fromdeltal(2, lmax, delta_ell=2)
fisher.set_binning(bins)
```

**Option B** — Config file:
```yaml
delta_ell: 2
```

Spectra automatically inherits bins from Fisher when you pass `fisher=fisher`.

In [4]:
# Option A: Python API
delta_ell = 2
lmax = 8
bins = Bins.fromdeltal(2, lmax, delta_ell)

print(f"Bins: {bins.nbins} bins, delta_ell = {delta_ell}")
for i in range(bins.nbins):
    print(
        f"  Bin {i}: ell = [{bins.lmins[i]}, {bins.lmaxs[i]}], eff_ell = {bins.lbin[i]}"
    )

config_file_bin = resolve_config("tests/data/nside4/T/config.yaml")
fisher_bin = Fisher(config_file_bin)
fisher_bin.set_binning(bins)
fisher_bin.run()

# Spectra inherits bins from Fisher automatically
spectra_bin = Spectra(config_file_bin, fisher=fisher_bin)
spectra_bin.run()

F_bin = fisher_bin.get_fisher_matrix()
cl_bin = spectra_bin.get_power_spectra()
err_bin = spectra_bin.get_error_bars()
ell_bin = bins.lbin

print(f"\nBinned: {F_bin.shape[0]} bins")
print(f"Power spectra shape: {cl_bin.shape}")
print(f"Fisher matrix shape: {F_bin.shape}")

Bins: 3 bins, delta_ell = 2
  Bin 0: ell = [2, 3], eff_ell = 2.5
  Bin 1: ell = [4, 5], eff_ell = 4.5
  Bin 2: ell = [6, 7], eff_ell = 6.5
2026-04-14 23:00:33 | fisher       | INFO     | Starting Fisher matrix analysis pipeline


2026-04-14 23:00:33 | fisher       | INFO     | --------------------------------------------------------------------------------
2026-04-14 23:00:33 | fisher       | INFO     | Fisher matrix computation completed

Binned: 3 bins
Power spectra shape: (50, 3)
Fisher matrix shape: (3, 3)


## 4. Linearity Check: F_binned == P @ F_unbinned @ P^T

By linearity of the trace, the natively binned Fisher matrix equals
the post-hoc binned version. This is an exact mathematical identity.

In [ ]:
# Build the binning matrix P (nbins x n_ell)
P_full, _ = bins._bin_operators()
n_ell = F_unb.shape[0]
n_ell_P = P_full.shape[1] - 2
P_ell = np.zeros((bins.nbins, n_ell))
P_ell[:, :n_ell_P] = P_full[:, 2:]

# Post-hoc binning
F_posthoc = P_ell @ F_unb @ P_ell.T

rel_diff = np.abs(F_bin - F_posthoc) / np.abs(F_posthoc)
print("Fisher linearity check:")
print(f"  Max relative difference: {np.max(rel_diff):.2e}")
print("  (should be < 1e-6)")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im = axes[0].imshow(F_bin, aspect="auto")
axes[0].set_title("Native binned Fisher")
plt.colorbar(im, ax=axes[0])

im = axes[1].imshow(F_posthoc, aspect="auto")
axes[1].set_title(r"Post-hoc: $P F P^T$")
plt.colorbar(im, ax=axes[1])

residual = F_bin - F_posthoc
im = axes[2].imshow(residual, aspect="auto")
axes[2].set_title("Residual")
plt.colorbar(im, ax=axes[2])

fig.tight_layout()
plt.show()

## 5. Compare Power Spectra

Plot unbinned and binned power spectra estimates with error bars.

In [ ]:
# Mean over simulations
cl_mean_unb = np.mean(cl_unb, axis=0)
cl_mean_bin = np.mean(cl_bin, axis=0)

# Bin half-widths for horizontal error bars
xerr_bin = bins.dl / 2.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Power spectra
ax = axes[0]
ax.errorbar(
    ell_unb,
    cl_mean_unb,
    yerr=err_unb,
    fmt="o-",
    ms=4,
    label=f"Unbinned ({len(ell_unb)} ells)",
    zorder=1,
)
ax.errorbar(
    ell_bin,
    cl_mean_bin,
    yerr=err_bin,
    xerr=xerr_bin,
    fmt="s",
    ms=6,
    capsize=3,
    label=f"Binned ({bins.nbins} bins, $\Delta\ell$={delta_ell})",
    zorder=2,
)
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$C_\ell$")
ax.set_title("QML Power Spectra")
ax.legend()

# Error bars comparison
ax = axes[1]
ax.plot(ell_unb, err_unb, "o-", ms=4, label="Unbinned", zorder=1)
ax.errorbar(
    ell_bin,
    err_bin,
    xerr=xerr_bin,
    fmt="s",
    ms=6,
    capsize=3,
    label=f"Binned ($\Delta\ell$={delta_ell})",
    zorder=2,
)
for i in range(bins.nbins):
    ax.axvspan(
        bins.lmins[i] - 0.5,
        bins.lmaxs[i] + 0.5,
        alpha=0.08,
        color="C1",
    )
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$\sigma(C_\ell)$")
ax.set_title("Error Bars")
ax.legend()
ax.set_yscale("log")

fig.tight_layout()
plt.show()

# Print comparison table
print("\nError bar comparison:")
header = f"{'Bin':>4} {'Eff ell':>8} {'Binned':>10} {'Unbinned':>16}"
print(header)
for i in range(bins.nbins):
    mask = (ell_unb >= bins.lmins[i]) & (ell_unb <= bins.lmaxs[i])
    unb_str = str(np.round(err_unb[mask], 1))
    print(f"{i:>4} {bins.lbin[i]:>8.1f} {err_bin[i]:>10.1f} {unb_str:>16}")

## 6. Normalization Modes with Binning

All three modes (deconvolved, decorrelated, convolved) work with binned quantities.

In [7]:
cl_deconv = spectra_bin.get_power_spectra(mode="deconvolved")
cl_decorr = spectra_bin.get_power_spectra(mode="decorrelated")
y, W, convolve = spectra_bin.get_power_spectra(mode="convolved")

cov_deconv = spectra_bin.get_covariance(mode="deconvolved")
cov_decorr = spectra_bin.get_covariance(mode="decorrelated")
cov_conv = spectra_bin.get_covariance(mode="convolved")

print(f"Deconvolved:  spectra {cl_deconv.shape}, covariance {cov_deconv.shape}")
print(f"Decorrelated: spectra {cl_decorr.shape}, covariance {cov_decorr.shape}")
print(f"Convolved:    y {y.shape}, W {W.shape}, covariance {cov_conv.shape}")
print("\nDecorrelated covariance (should be identity):")
print(np.round(cov_decorr, 10))

Deconvolved:  spectra (50, 3), covariance (3, 3)
Decorrelated: spectra (50, 3), covariance (3, 3)
Convolved:    y (50, 3), W (3, 3), covariance (3, 3)

Decorrelated covariance (should be identity):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 7. Config File API

Binning can also be configured via the YAML config file:

```yaml
delta_ell: 2
```

This is equivalent to calling `fisher.set_binning(Bins.fromdeltal(2, lmax, 2))`.

In [8]:
config_with_binning = resolve_config(
    "tests/data/nside4/T/config.yaml",
    overrides={"delta_ell": 2},
)

fisher_cfg = Fisher(config_with_binning)
fisher_cfg.run()

spectra_cfg = Spectra(config_with_binning, fisher=fisher_cfg)
spectra_cfg.run()

F_cfg = fisher_cfg.get_fisher_matrix()
print(f"Config-based binning: Fisher shape = {F_cfg.shape}")
print(f"Matches Python API: {np.allclose(F_cfg, F_bin)}")

2026-04-14 23:00:35 | fisher       | INFO     | Starting Fisher matrix analysis pipeline


2026-04-14 23:00:36 | fisher       | INFO     | --------------------------------------------------------------------------------
2026-04-14 23:00:36 | fisher       | INFO     | Fisher matrix computation completed
Config-based binning: Fisher shape = (3, 3)
Matches Python API: True


## 8. Cleanup

In [ ]:
# for f in [config_file, config_file_bin, config_with_binning]:
#     if os.path.exists(f):
#         os.unlink(f)
# print("Temp files cleaned up.")